In [ ]:
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
except ImportError as error:
    raise ImportError(
        "Install PyTorch in the active environment before running this section."
    ) from error

torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", DEVICE)

In [ ]:
# RNNs use raw operating context and sensor trajectories, not engineered rolling features.
sequence_features = [
    column
    for column in setting_cols + selected_sensors
    if column not in low_variance_cols
]

sequence_train = train[
    ["unit_id", "cycle", "rul_capped"] + sequence_features
].copy()
sequence_test = test_raw[
    ["unit_id", "cycle"] + sequence_features
].copy()

# Reuse exactly the same engine split used by the tabular models.
sequence_train_part = sequence_train[
    sequence_train["unit_id"].isin(train_engine_ids)
].copy()
sequence_val_part = sequence_train[
    sequence_train["unit_id"].isin(val_engine_ids)
].copy()

# Fit the scaler on development-training engines only.
sequence_scaler = StandardScaler()
sequence_scaler.fit(sequence_train_part[sequence_features])

sequence_train_part.loc[:, sequence_features] = (
    sequence_scaler.transform(sequence_train_part[sequence_features])
)
sequence_val_part.loc[:, sequence_features] = (
    sequence_scaler.transform(sequence_val_part[sequence_features])
)

print("Sequence features:", sequence_features)


In [ ]:
def build_train_sequences(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    sequence_length: int,
):
    X, y, unit_ids = [], [], []

    for unit_id, engine in df.groupby("unit_id"):
        engine = engine.sort_values("cycle")
        values = engine[features].to_numpy(dtype=np.float32)
        targets = engine[target].to_numpy(dtype=np.float32)

        if len(engine) < sequence_length:
            continue

        for end in range(sequence_length, len(engine) + 1):
            start = end - sequence_length
            X.append(values[start:end])
            y.append(targets[end - 1])
            unit_ids.append(unit_id)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        np.asarray(unit_ids),
    )


X_train_seq, y_train_seq, train_seq_units = build_train_sequences(
    sequence_train_part,
    sequence_features,
    "rul_capped",
    SEQUENCE_LENGTH,
)
X_val_seq, y_val_seq, val_seq_units = build_train_sequences(
    sequence_val_part,
    sequence_features,
    "rul_capped",
    SEQUENCE_LENGTH,
)

print("Train sequences:", X_train_seq.shape)
print("Validation sequences:", X_val_seq.shape)

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(
            X.astype(np.float32, copy=False)
        )
        self.y = torch.from_numpy(
            y.astype(np.float32, copy=False)
        ).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


train_loader = DataLoader(
    SequenceDataset(X_train_seq, y_train_seq),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    SequenceDataset(X_val_seq, y_val_seq),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

In [ ]:
class RULRecurrentModel(nn.Module):
    """One configurable implementation for both LSTM and GRU experiments."""

    def __init__(
        self,
        input_size,
        rnn_type="GRU",
        hidden_size=64,
        num_layers=1,
        dropout=0.20,
    ):
        super().__init__()

        rnn_type = rnn_type.upper()
        recurrent_class = {
            "LSTM": nn.LSTM,
            "GRU": nn.GRU,
        }.get(rnn_type)

        if recurrent_class is None:
            raise ValueError("rnn_type must be 'LSTM' or 'GRU'.")

        self.rnn_type = rnn_type
        self.recurrent = recurrent_class(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.regressor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        output, _ = self.recurrent(x)
        return self.regressor(output[:, -1, :])

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)

    total_loss = 0.0
    total_examples = 0

    with torch.set_grad_enabled(is_training):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            if is_training:
                optimizer.zero_grad()

            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            if is_training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )
                optimizer.step()

            batch_size = X_batch.size(0)
            total_loss += loss.item() * batch_size
            total_examples += batch_size

    return total_loss / total_examples


def predict_loader(model, loader):
    model.eval()
    predictions, actuals = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            predictions.append(
                model(X_batch.to(DEVICE)).cpu().numpy()
            )
            actuals.append(y_batch.numpy())

    return (
        np.concatenate(actuals).ravel(),
        np.concatenate(predictions).ravel(),
    )

In [ ]:
def run_recurrent_experiment(config: dict) -> dict:
    """Train one architecture with early stopping and return all artifacts."""
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)

    model = RULRecurrentModel(
        input_size=len(sequence_features),
        rnn_type=config["rnn_type"],
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
    ).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.get("learning_rate", 1e-3),
        weight_decay=config.get("weight_decay", 1e-5),
    )

    history = {"train_loss": [], "val_loss": []}
    best_state = None
    best_val_loss = np.inf
    best_epoch = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
        )
        val_loss = run_epoch(
            model,
            val_loader,
            criterion,
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"{config['name']} | Epoch {epoch:02d} | "
            f"Train RMSE {train_loss ** 0.5:.3f} | "
            f"Val RMSE {val_loss ** 0.5:.3f}"
        )

        if epochs_without_improvement >= PATIENCE:
            print(f"{config['name']}: early stopping.")
            break

    if best_state is None:
        raise RuntimeError("No model state was saved.")

    model.load_state_dict(best_state)
    y_actual, y_pred = predict_loader(model, val_loader)
    y_pred = np.clip(y_pred, 0, RUL_CAP)

    result = evaluate_regression(
        y_actual,
        y_pred,
        config["name"],
    )
    result.update({
        "rnn_type": config["rnn_type"],
        "hidden_size": config["hidden_size"],
        "num_layers": config["num_layers"],
        "dropout": config["dropout"],
        "best_epoch": best_epoch,
    })

    return {
        "model": model,
        "history": history,
        "result": result,
        "config": config,
    }


In [ ]:
# Same architecture search space for LSTM and GRU.
recurrent_configs = [
    {
        "name": "LSTM_64x1",
        "rnn_type": "LSTM",
        "hidden_size": 64,
        "num_layers": 1,
        "dropout": 0.20,
    },
    {
        "name": "LSTM_96x2",
        "rnn_type": "LSTM",
        "hidden_size": 96,
        "num_layers": 2,
        "dropout": 0.25,
    },
    {
        "name": "LSTM_128x2",
        "rnn_type": "LSTM",
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.30,
    },
    {
        "name": "GRU_64x1",
        "rnn_type": "GRU",
        "hidden_size": 64,
        "num_layers": 1,
        "dropout": 0.20,
    },
    {
        "name": "GRU_96x2",
        "rnn_type": "GRU",
        "hidden_size": 96,
        "num_layers": 2,
        "dropout": 0.25,
    },
    {
        "name": "GRU_128x2",
        "rnn_type": "GRU",
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.30,
    },
]

recurrent_experiments = {}

for config in recurrent_configs:
    print("\n" + "=" * 72)
    print("Training:", config["name"])
    print("=" * 72)
    recurrent_experiments[config["name"]] = (
        run_recurrent_experiment(config)
    )

In [ ]:
recurrent_comparison = (
    pd.DataFrame([
        experiment["result"]
        for experiment in recurrent_experiments.values()
    ])
    .sort_values(["cmapss_score", "rmse", "mae"])
    .reset_index(drop=True)
)

display(
    recurrent_comparison.style.format({
        "rmse": "{:.2f}",
        "mae": "{:.2f}",
        "cmapss_score": "{:,.0f}",
        "dropout": "{:.2f}",
    })
)

best_lstm_row = (
    recurrent_comparison[
        recurrent_comparison["rnn_type"] == "LSTM"
    ]
    .sort_values(["cmapss_score", "rmse"])
    .iloc[0]
)
best_gru_row = (
    recurrent_comparison[
        recurrent_comparison["rnn_type"] == "GRU"
    ]
    .sort_values(["cmapss_score", "rmse"])
    .iloc[0]
)

best_lstm_config = recurrent_experiments[
    best_lstm_row["model"]
]["config"]
best_gru_config = recurrent_experiments[
    best_gru_row["model"]
]["config"]

best_lstm_epoch = int(best_lstm_row["best_epoch"])
best_gru_epoch = int(best_gru_row["best_epoch"])

print("Selected LSTM:", best_lstm_config, "Epochs:", best_lstm_epoch)
print("Selected GRU:", best_gru_config, "Epochs:", best_gru_epoch)

# Select the recurrent model family using validation results only.
selected_recurrent_model_name = recurrent_comparison.iloc[0]["model"]

print("Validation-selected recurrent model:", selected_recurrent_model_name)


In [ ]:
# Plot training and validation RMSE for the best LSTM and GRU configurations.
for model_name in [best_lstm_row["model"], best_gru_row["model"]]:
    history = recurrent_experiments[model_name]["history"]

    plt.figure(figsize=(8, 4))
    plt.plot(
        np.sqrt(history["train_loss"]),
        label="Training RMSE",
    )
    plt.plot(
        np.sqrt(history["val_loss"]),
        label="Validation RMSE",
    )
    plt.xlabel("Epoch")
    plt.ylabel("RMSE")
    plt.title(f"{model_name}: Training vs Validation")
    plt.legend()
    plt.tight_layout()
    save_current_figure(f"{model_name.lower()}_learning_curve.png")
    plt.show()
